## Imports

In [ ]:
%matplotlib inline


In [ ]:
import os
from pathlib import Path

In [ ]:
import SimpleITK as sitk
import nibabel as nib
import numpy as np
import pandas as pd
from scipy import ndimage

In [ ]:
from utils.dataset import initialization_dict
from utils.matching import get_lesion_track
from utils.preprocessing import preprocess_mask
from utils.graphics import explore_3D_array, explore_patient_timepoints, visualize_matching, visualize_track_overlap

from utils.tables import build_components, build_tracks, build_unmatched, build_volume_change, build_summary

## Dictionary - File system 

In [ ]:
base_path = Path("dataset/")

dataset_dict = initialization_dict(base_path)

## Patient selection and image verification

In [ ]:
patient_id = 'P1'

sample_file_path = str(dataset_dict[patient_id]['T1']['T2'])
sitk_image = sitk.ReadImage(sample_file_path)
image_array = sitk.GetArrayFromImage(sitk_image)
print(f"list dim: {image_array.shape}")

In [ ]:
explore_3D_array(image_array)

## Load and Preprocessing Masks

In [ ]:
patient_data = dataset_dict[patient_id]
timepoints = sorted(patient_data.keys())

masks = {}
for tp in timepoints:
    key = next((k for k in patient_data[tp] if k.upper() == "MASK"), None)
    if key:
        arr = sitk.GetArrayFromImage(sitk.ReadImage(str(patient_data[tp][key])))
        masks[tp] = (arr > 0).astype(np.uint8)

print(masks.keys())


In [ ]:
masks = {tp: preprocess_mask(masks[tp]) for tp in timepoints}   # mask preprocessing

In [ ]:
explore_patient_timepoints(dataset_dict, patient_id=patient_id, modality='MASK', cmap='gray')

## Isolation of individual lesions

In [ ]:
labeled = {}
n_components = {}

for tp, mask in masks.items():
    lbl, n = ndimage.label(masks[tp])
    labeled[tp] = lbl
    n_components[tp] = n

    total_voxels = (lbl > 0).sum()

    print(f'{tp}: finded components = {n}, voxels: {total_voxels}')


## Component Table
collect the properties of each source: volume, centroid, bounding box, and component number.

In [ ]:
components_table = build_components(
    timepoints=timepoints,
    labeled=labeled,
    n_components=n_components,
)

display(components_table)

# t1_table = components_table[components_table['timepoint'] == 'T1']

# display(t1_table)

## Matching and source trajectories
match sources between neighbor timepoints and draw their trajectories.

In [ ]:
masks.keys()

In [ ]:
tracks, matching_debug_tables = get_lesion_track(
    timepoints=timepoints,
    labeled=labeled,
    n_components=n_components,
    spacing=(1.0, 1.0, 1.0),
    centroid_threshold_mm=3.0,
    surface_threshold_mm=2.0,
    min_score=-0.5,
    return_tables=True,
)

tracks_table = build_tracks(
    timepoints=timepoints,
    tracks=tracks,
)

display(tracks_table)

## New and Disappeared Lesions
show which components appeared or disappeared between adjacent timepoints.

In [ ]:
unmatched_table = build_unmatched(
    timepoints=timepoints,
    tracks=tracks,
    n_components=n_components,
)

display(unmatched_table)

## Change in lesion volume
classify lesions as increased, decreased, or stable.

In [ ]:
volume_change_table = build_volume_change(
    timepoints=timepoints,
    tracks=tracks,
    labeled=labeled,
    relative_threshold=0.25,
    absolute_threshold_voxels=5,
)

display(volume_change_table)

## Summary table for verification
compile a summary for each lesion: status, volume, matching quality, and attention flag.



In [ ]:
summary_table = build_summary(
    timepoints=timepoints,
    tracks=tracks,
    labeled=labeled,
    matching_debug_tables=matching_debug_tables,
    spacing=(1.0, 1.0, 1.0),
    growth_relative_threshold=0.25,
    growth_absolute_threshold_mm3=5.0,
    new_lesion_attention_threshold_voxels=200,
)

display(summary_table)

## Overview tracking visualization

- green: the lesion is associated with a neighboring timepoint;
- orange: a new lesion that did not appear at the first timepoint;
- red: the lesion disappeared before the last timepoint;
- purple: an isolated lesion that exists only at one intermediate timepoint;
- gray: other components not included in the main categories.

In [ ]:
visualize_matching(labeled, tracks.values())

## Checking a specific track
compare one lesion source between neighbor timepoints:
- green: the lesion area at the first timepoint;
- red: the lesion area at the second timepoint;
- yellow: the intersection of the two masks.

In [ ]:
visualize_track_overlap(labeled, tracks)